In [ ]:
import os
os.environ["SESSION_ID_FILE"] = "Files/ci-artifacts/livy-session-id-interactive.txt"
# Parameters — update before running interactively
ci_target = "ephemeral_ci"
prod_state_path = "abfss://WORKSPACE_ID@onelake.dfs.fabric.microsoft.com/LAKEHOUSE_ID/Files/ci-artifacts/prod-state"
repo_url = "https://github.com/ORG/REPO.git"
repo_branch = "BRANCH"
github_app_id = "GH-APP-ID"
github_installation_id = "GH-APP-INSTALLATION-ID"
github_pem_secret = "GH-APP-PEM"
vault_url = "https://KV-NAME.vault.azure.net/"
lakehouse_name = "LAKEHOUSE_NAME"
lakehouse_id = "LAKEHOUSE_ID"
workspace_id = "WORKSPACE_ID"
workspace_name = "WORKSPACE_NAME"
schema_name = "dbo"
run_mode = "interactive"


In [ ]:
# Download prod-state manifest from OneLake (ABFSS → local path)
# No-op when prod_state_path is already a local path (e.g. greenfield ./prod-state).
if prod_state_path.startswith('abfss://'):
    import os
    import urllib.request
    https_url = prod_state_path.replace('abfss://', 'https://onelake.dfs.fabric.microsoft.com/', 1)
    https_url = https_url.replace('@onelake.dfs.fabric.microsoft.com', '', 1)
    manifest_url = https_url.rstrip('/') + '/manifest.json'
    local_dir = '/tmp/prod-state'
    os.makedirs(local_dir, exist_ok=True)
    token = notebookutils.credentials.getToken('storage')  # noqa: F821
    req = urllib.request.Request(manifest_url, headers={'Authorization': f'Bearer {token}'})
    with urllib.request.urlopen(req) as resp:
        with open(f'{local_dir}/manifest.json', 'wb') as f:
            f.write(resp.read())
    local_prod_state_path = local_dir
else:
    local_prod_state_path = prod_state_path

In [ ]:
# Command assembly
_deps  = "dbt deps"
_clone = f"dbt clone --select state:modified+ --exclude config.materialized:view --state {local_prod_state_path} --target-path target/clone --profiles-dir .github/profiles --target {ci_target}"
run = f"dbt run --select state:modified+ --defer --state {local_prod_state_path} --target-path target/build --profiles-dir .github/profiles --target {ci_target}"
_empty_build = f"dbt build --select state:modified+ --empty --profiles-dir .github/profiles --target {ci_target}"
# Build unit test command from deployment manifest — indirect selection scopes to state:modified+ models.
# dbt unit test nodes are not reachable via state:modified+ graph traversal;
# naming models directly triggers indirect selection which includes their unit tests.
import json as _json  # noqa: E402
_manifest_path = f"/lakehouse/default/Files/ci-artifacts/deployment-manifest-{head_sha}.json"  # noqa: F821
_manifest_models = [
    a["name"].split(".")[-1]
    for a in _json.load(open(_manifest_path)).get("artifacts", [])
    if a.get("materialized") != "ephemeral"
]
_unit = (
    f"dbt test --select {' '.join(_manifest_models)},test_type:unit"
    f" --target-path target/unit --profiles-dir .github/profiles --target {ci_target}"
    if _manifest_models else
    f"dbt test --select test_type:unit"
    f" --target-path target/unit --profiles-dir .github/profiles --target {ci_target}"
)
_data  = f"dbt test --select state:modified+ --exclude test_type:unit --store-failures --state {local_prod_state_path} --target-path target/data --profiles-dir .github/profiles --target {ci_target}"

clone_command     = [_deps, _clone]
build_command     = [_deps, run]
unit_test_command = [_deps, _empty_build, _unit]
data_test_command = [_deps, _data]


In [ ]:
!pip install vd-dbt-fabricspark==1.9.15 -q

In [ ]:
from dbt.adapters.fabricspark.notebook import (
    run_dbt_job,
    DbtJobConfig,
    RepoConfig,
    ConnectionConfig,
)

In [ ]:
# Clone (interactive)
config = DbtJobConfig(
    command=clone_command,
    repo=RepoConfig(
        url=repo_url, branch=repo_branch,
        github_app_id=github_app_id, github_installation_id=github_installation_id,
        github_pem_secret=github_pem_secret, vault_url=vault_url,
    ),
    connection=ConnectionConfig(
        lakehouse_name=lakehouse_name, lakehouse_id=lakehouse_id,
        workspace_id=workspace_id, workspace_name=workspace_name, schema_name=schema_name,
    ),
)
result = run_dbt_job(config)

In [ ]:
# Build (interactive)
config = DbtJobConfig(
    command=build_command,
    repo=RepoConfig(
        url=repo_url, branch=repo_branch,
        github_app_id=github_app_id, github_installation_id=github_installation_id,
        github_pem_secret=github_pem_secret, vault_url=vault_url,
    ),
    connection=ConnectionConfig(
        lakehouse_name=lakehouse_name, lakehouse_id=lakehouse_id,
        workspace_id=workspace_id, workspace_name=workspace_name, schema_name=schema_name,
    ),
)
result = run_dbt_job(config)

In [ ]:
# Unit Test (interactive)
config = DbtJobConfig(
    command=unit_test_command,
    repo=RepoConfig(
        url=repo_url, branch=repo_branch,
        github_app_id=github_app_id, github_installation_id=github_installation_id,
        github_pem_secret=github_pem_secret, vault_url=vault_url,
    ),
    connection=ConnectionConfig(
        lakehouse_name=lakehouse_name, lakehouse_id=lakehouse_id,
        workspace_id=workspace_id, workspace_name=workspace_name, schema_name=schema_name,
    ),
)
result = run_dbt_job(config)

In [ ]:
# Data Test (interactive)
config = DbtJobConfig(
    command=data_test_command,
    repo=RepoConfig(
        url=repo_url, branch=repo_branch,
        github_app_id=github_app_id, github_installation_id=github_installation_id,
        github_pem_secret=github_pem_secret, vault_url=vault_url,
    ),
    connection=ConnectionConfig(
        lakehouse_name=lakehouse_name, lakehouse_id=lakehouse_id,
        workspace_id=workspace_id, workspace_name=workspace_name, schema_name=schema_name,
    ),
)
result = run_dbt_job(config)